In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal

In [2]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
@tool
def calculator(expression: str) -> str:
    """Calculate a math expression. Use for any arithmetic. Args: expression: e.g., '25 * 47'"""
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search_web(query: str) -> str:
    """Search the web for information. Use for current events or facts. Args: query: search terms"""
    return f"Search results for '{query}': Several major developments reported this week."


In [4]:
tools = [calculator, search_web]
llm_with_tools = llm.bind_tools(tools)

In [5]:
tools_by_name = {t.name: t for t in tools}

In [6]:
print(tools_by_name)

{'calculator': StructuredTool(name='calculator', description="Calculate a math expression. Use for any arithmetic. Args: expression: e.g., '25 * 47'", args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x000001E145BA5E40>), 'search_web': StructuredTool(name='search_web', description='Search the web for information. Use for current events or facts. Args: query: search terms', args_schema=<class 'langchain_core.utils.pydantic.search_web'>, func=<function search_web at 0x000001E145BA6520>)}


In [ ]:
class State(TypedDict):
    # Appending Field
    messages: Annotated[list, add_messages]
    # Overwriting Fields
    user_name: str                            
    step_count: int                            
    is_approved: bool  

In [8]:
def llm_node(state: State) -> dict:
    """
    The 'thinking' node. The LLM looks at the conversation and decides:
    - Should I call a tool? → generates a tool_call
    - Should I respond directly? → generates a text response
    """
    system = SystemMessage(
        content="You are a helpful assistant. Use tools when needed to get accurate information."
    )
    
    response = llm_with_tools.invoke([system] + state["messages"])
    
    return {"messages": [response]} 

In [9]:
def tool_node(state: State) -> dict:
    """
    The 'acting' node. Executes whatever tools the LLM requested
    and returns the results as ToolMessages.
    """
    results = []
    
   
    last_message = state["messages"][-1]
    
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]              
        tool_args = tool_call["args"]              
        
        
        tool_result = tools_by_name[tool_name].invoke(tool_args)
        
      
        results.append(ToolMessage(
            content=str(tool_result),             
            tool_call_id=tool_call["id"]           
        ))
    
    return {"messages": results} 

In [10]:
def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    """
    After the LLM responds, check: did it call any tools?
    If YES → go to tool_node to execute them
    If NO  → go to END (the LLM gave a final text answer)
    """
    last_message = state["messages"][-1]
    
  
    if last_message.tool_calls:
        return "tool_node"
    
   
    return "__end__"

In [11]:
graph_builder = StateGraph(State)

In [12]:
graph_builder.add_node("llm_node", llm_node)      
graph_builder.add_node("tool_node", tool_node)  

In [13]:
graph_builder.add_edge(START, "llm_node") 

In [14]:
graph_builder.add_conditional_edges(
    "llm_node",                                      
    should_continue,                                  
    ["tool_node", "__end__"]                          
)

In [15]:
graph_builder.add_edge("tool_node", "llm_node")  

In [16]:
agent = graph_builder.compile()

In [21]:
result = agent.invoke({
    "messages": [HumanMessage(content="What is 25 * 47 and what's the latest information about russia war with ukraine?")]
})

In [22]:
for msg in result["messages"]:
    if hasattr(msg, 'content') and msg.content:
        print(f"[{msg.__class__.__name__}]: {msg.content}\n")


[HumanMessage]: What is 25 * 47 and what's the latest information about russia war with ukraine?

[ToolMessage]: 25 * 47 = 1175

[ToolMessage]: Search results for 'latest news Russia Ukraine war': Several major developments reported this week.

[AIMessage]: The result of \( 25 \times 47 \) is \( 1175 \).

Regarding the latest information about the Russia-Ukraine war, several major developments have been reported this week. If you would like more specific details or a summary of those developments, please let me know!

